# Gear Quality Model Training (Improved Accuracy)

This notebook trains an SVM classifier for `bad` vs `good` gears from images stored in:

```
dataset/
  bad/
  good/
```

Improvements vs the basic version:
- CLAHE + light denoise preprocessing (more robust to lighting)
- `StandardScaler` + `SVC` pipeline (recommended for SVMs)
- Hyperparameter tuning with `GridSearchCV` (linear vs RBF kernels)
- `class_weight='balanced'` and stratified split (reduces defective parts passing as good)


In [1]:
import os
import cv2
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [2]:
# ---------------- Configuration ----------------
DATASET_PATH = 'dataset'
CATEGORIES = ['bad', 'good']   # folder names
IMG_SIZE = 100
MODEL_NAME = 'gear_model.pkl'


In [3]:
def preprocess(img_gray):
    """Resize + contrast normalization to reduce lighting variation."""
    img = cv2.resize(img_gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # CLAHE helps when lighting/exposure changes across images
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # light denoise
    img = cv2.GaussianBlur(img, (3, 3), 0)
    return img


def load_data():
    data, labels = [], []

    for category in CATEGORIES:
        path = os.path.join(DATASET_PATH, category)
        class_num = CATEGORIES.index(category)

        if not os.path.exists(path):
            print(f"Warning: Folder '{path}' not found.")
            continue

        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            if not os.path.isfile(img_path):
                continue

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            img = preprocess(img)
            data.append(img.flatten())
            labels.append(class_num)

    return np.asarray(data), np.asarray(labels)


In [4]:
print('Loading and processing images...')
X, y = load_data()

print('bad:', len([p for p in os.listdir(os.path.join(DATASET_PATH, 'bad'))]) if os.path.exists(os.path.join(DATASET_PATH, 'bad')) else 0)
print('good:', len([p for p in os.listdir(os.path.join(DATASET_PATH, 'good'))]) if os.path.exists(os.path.join(DATASET_PATH, 'good')) else 0)

if len(X) == 0:
    raise SystemExit('No images found! Please add images to dataset/good and dataset/bad.')

print('X shape:', X.shape, 'y shape:', y.shape)
print('Class counts:', {c: int((y==i).sum()) for i,c in enumerate(CATEGORIES)})


Loading and processing images...
bad: 35
good: 23
X shape: (58, 10000) y shape: (58,)
Class counts: {'bad': 35, 'good': 23}


In [5]:
# ✅ Stratified split keeps class distribution similar in train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)


Train: (46, 10000) Test: (12, 10000)


In [6]:
print('Training + tuning SVM (GridSearchCV)...')

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, class_weight='balanced', random_state=42)),
])

param_grid = [
    {'svc__kernel': ['linear'], 'svc__C': [0.1, 1, 10, 100]},
    {'svc__kernel': ['rbf'], 'svc__C': [0.1, 1, 10, 100], 'svc__gamma': ['scale', 'auto', 0.01, 0.001]},
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)
model = search.best_estimator_

print('Best params:', search.best_params_)
print('Best CV f1_macro:', f'{search.best_score_:.4f}')


Training + tuning SVM (GridSearchCV)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params: {'svc__C': 1, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
Best CV f1_macro: 0.7564


In [7]:
print('Evaluating on test set...')
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print('Test accuracy:', f'{acc:.4f}')
print('Confusion matrix:\n', confusion_matrix(y_test, preds))
print('\nClassification report:\n')
print(classification_report(y_test, preds, target_names=CATEGORIES))


Evaluating on test set...
Test accuracy: 0.6667
Confusion matrix:
 [[5 2]
 [2 3]]

Classification report:

              precision    recall  f1-score   support

         bad       0.71      0.71      0.71         7
        good       0.60      0.60      0.60         5

    accuracy                           0.67        12
   macro avg       0.66      0.66      0.66        12
weighted avg       0.67      0.67      0.67        12



In [8]:
joblib.dump(model, MODEL_NAME)
print(f'Model successfully saved as {MODEL_NAME}!')


Model successfully saved as gear_model.pkl!


## Next step: reduce defective parts passing
If you want to be stricter at inference time, only allow PASS when the probability of **good** is high (e.g., 0.90 or 0.95).
That logic should live in `predict.py` and/or `appgi.py`, not in training.
